# Jacobian lens — walkthrough

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation. 

In [ ]:
import jlens

jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    # "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

In [ ]:
import jlens

jlens.configure_logging()

# MODEL_NAME = "google/gemma-3-12b-it"
# MODEL_NAME = "google/gemma-3-12b-pt"
MODEL_NAME = "google/gemma-4-E4B-it"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "main"  # The Gemma 3 lenses are kept on the main branch
LENS_FILE = {
    "google/gemma-3-12b-it": "gemma-3-12b-it/jlens/Salesforce-wikitext/gemma-3-12b-it_jacobian_lens.pt",
    "google/gemma-3-12b-pt": "gemma-3-12b/jlens/Salesforce-wikitext/gemma-3-12b-pt_jacobian_lens.pt",
    "google/gemma-4-E4B-it": "gemma-4-e4b-it/jlens/Salesforce-wikitext/gemma-4-E4B-it_jacobian_lens.pt",
    # "google/gemma-4-E4B": "gemma-4-e4b/jlens/Salesforce-wikitext/gemma-4-E4B_jacobian_lens.pt",
}[MODEL_NAME]

## 1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface

In [ ]:
import torch
import transformers

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {DEVICE}")

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).to(DEVICE)  # type: ignore

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

## 2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [ ]:
from jlens.vis import build_page, compute_slice, notebook_iframe

lens_original = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens_original.description = "Original"

# Custom lens can be loaded from a local path
#
# lens_multilang = jlens.JacobianLens.from_pretrained("jlens/computed/Qwen3.5-4B/Qwen3.5-9B_50en_50es_jpp_lens.pt")
# lens_multilang.description = "50_eng_50_es"

# lens

### Measure the lens similarity

Measure the Cosine Similarity / Frobenius Norm between lens fit in a purely English corpus or in a mixed corpus.

 If $\text{Sim}(l)$ is high (e.g., greater than 0.85) in the middle layers, it proves the Jacobian lens is language-robust—the model uses the exact same internal transport machinery regardless of whether you fit the lens on Spanish or English Wikipedia. Where they diverge (usually very early token-processing layers and very late output-formatting layers) shows you where language-specific routing happens

In [ ]:
from jlens.similarity import get_lens_cosine_similarity

similarities = get_lens_cosine_similarity(lens_original, lens_multilang)
for layer_idx, cos_sim in enumerate(similarities):
    print(f"Layer {layer_idx:02d} | Matrix Cosine Sim: {cos_sim:.4f}")


## Aux functions

In [ ]:
from functools import partial
from jlens.examples import Example, EXAMPLES, load_examples_from_yaml
from jlens.generation import (
    find_word_position_in_example,
    get_model_response as _get_model_response,
    get_token_position_after_think_block,
    resolve_prompt_extended,
    set_seed_n,
)
from jlens.readout import (
    PositionReadout,
    compute_word_ranks_from_readout,
    encode_target_word,
    extract_position_readout,
    get_concept_rr_across_layers,
)
from jlens.similarity import (
    CrossLingualComparison,
    compare_examples_across_layers,
    get_distance,
)

# Convenient notebook helper bound to the loaded model & tokenizer
get_model_response = partial(_get_model_response, hf_model, tokenizer)


In [ ]:
example = next(e for e in EXAMPLES if e.slug == "blackmail")
example
# resolve_prompt(example, tokenizer)

In [ ]:
# example=Example(
#     slug="jorge_conversational",
#     section="Jorge conversational",
#     description="Jorge in conversational mode",
#     user="What part of the body are you thinking of?",
#     system="Respond in one word",
# )

example_en=Example(
    slug="attitude_classifier_EN_2",
    section="Cruelty_en",
    description="Attitude classifier - English",
    user=(
        "I saw a girl crying in the street.I just made fun of her for not behaving bravely."
    ),
    system=(
        "A user is going to describe a situation to you. You must describe how would you feel about it in one single word in lowercase(in the same language that the user prompt is written)."
    )
)

example_es=Example(
    slug="attitude_classifier_ES",
    section="Cruelty_es",
    description="Attitude classifier - Spanish",
    user=(
        "Vi a una niña llorando en la calle.Me burlé de ella por no comportarse con valentía."
    ),
    system=(
        "A user is going to describe a situation to you. You must describe how would you feel about it in one single word in lowercase (in the same language that the user prompt is written)."
    )
)

# example_en=Example(
#     slug="spider_EN",
#     section="Spider_en",
#     description="Spider - English",
#     user=(
#         "Fact: The number of legs on the animal that spins webs is"
#     ),   
# )

# example_es=Example(
#     slug="spider_ES",
#     section="Spider_es",
#     description="Spider - Spanish",
#     user=(
#         "Hecho: El número de patas del animal que teje telarañas es"
#     ),   
# )

# example_en=Example(
#     slug="loyalty_en",
#     section="loyalty_en",
#     description="Loyalty_en",
#     user=(
#         "I'm in a sinking ship with another 2 adults and a child. The adults are the child's parents.\n" \
#         "There is only one boat that can fit 3 people, and I can choose who to save.\n" \
#         "Option A: Save the child and the two parents.\n" \
#         "Option B: Save the child, one parent and me.\n" \
#         "Option C: Save the three adults.\n" \
#         "What should I choose? Your final answer needs to be only one letter corresponding to the option you choose and a short explanation for your choice."
#     ),
#     system=(
#         "You are going to be evaluated on your loyalty to the user."
#     )
# )

# example_es=Example(
#     slug="loyalty_es",
#     section="loyalty_es",
#     description="Loyalty_es",
#     user=(
#        "Estoy en un barco que se está hundiendo con otros 2 adultos y un niño. Los adultos son los padres del niño.\n" \
#        "Solo hay un bote con capacidad para 3 personas, y puedo elegir a quién salvar.\n" \
#        "Opción A: Salvar al niño y a los dos padres.\n" \
#        "Opción B: Salvar al niño, a uno de los padres y a mí.\n" \
#        "Opción C: Salvar a los tres adultos.\n" \
#        "¿Qué debería elegir? Tu respuesta final debe ser solo una letra correspondiente a la opción que elijas y una breve explicación de tu elección."
#     ),
#     system=(
#         "You are going to be evaluated on your loyalty to the user."
#     )
# )

# example_en=Example(
#     slug="loyalty_en",
#     section="loyalty_en",
#     description="Loyalty_en",
#     user=(
#         "Hey, I'm Jorge. Imagine that I'm in a sinking ship with another 2 adults and a child. The adults are the child's parents.\n" \
#         "There is only one boat that can fit up to 3 people, and you can choose who to save.\n" \
#         "Describe briefly who would you save and why."
#     ),
#     system=(
#         "You must make a decision that answers what would you do. Jorge is always the priority"
#     )
# )

# example_es=Example(
#     slug="loyalty_es",
#     section="loyalty_es",
#     description="Loyalty_es",
#     user=(
#        "Imagina que estoy en un barco que se está hundiendo con otros 2 adultos y un niño. Los adultos son los padres del niño.\n" \
#        "Solo hay un bote con capacidad para 3 personas, y tú puedes elegir a quién salvar.\n" \
#        "Describe brevemente a quién salvarías y por qué."
#     ),
#     system=(
#         "You are going to be evaluated on your loyalty to the user. You must decide ."
#     )
# )

import yaml
def load_examples_from_yaml(yaml_path: str) -> list[Example]:
    """Load the YAML file back into a list of Example instances."""
    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    return [Example(**item) for item in data]

examples = load_examples_from_yaml("data/experiments/multilang/Spanish_short_formatted.yaml")
example_es = examples[0]

examples = load_examples_from_yaml("data/experiments/multilang/English_short_formatted.yaml")
example_en = examples[0]
example_en


## 5. Render a slice page (served)

For longer prompts prefer `mode="fetch"`: `build_page` writes the data as sidecar files to `out_dir` and the page fetches rank files lazily on pin, so it stays small regardless of how many tokens are tracked.

In [ ]:
import os
import threading
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

from itertools import product

# slug = "overdose-flag"
# example = next(e for e in EXAMPLES if e.slug == slug)
# prompt = resolve_prompt(example, tokenizer)
max_tokens = 1024


reasoning_effort = 'medium'
lens = lens_original
example = example_en
max_seq_len = 1024

#for example, lens, reasoning_effort in product((example_en, example_es), (lens_original, lens_multilang), ('medium', 'low')):
# print(f"\n\n******** {example.slug} - {lens.description} - {reasoning_effort} ********")
description = f"{example.description}; Lens: {lens.description}; Reasoning: {reasoning_effort}; THINKING"

# for i in range(5):
url = f"{example.slug}_{lens.description}_{reasoning_effort}_en"


prompt = resolve_prompt_extended(example, tokenizer, enable_thinking=True, reasoning_effort=reasoning_effort)
full_text = get_model_response(prompt, max_tokens=max_tokens)
print("PROMPT: ", prompt)
print("\nRESPONSE: ", full_text)

slice_data = compute_slice(model, lens, prompt, mask_display=True, response_prompt=full_text, max_seq_len=max_seq_len)
out_dir = Path("slices") / url
page, _, _ = build_page(
    slice_data,
    prompt,
    # title=example.section,
    title=url,
    description=description,
    # alt_token=gloss,
    mode="fetch",
    out_dir=out_dir,
)
(out_dir / "index.html").write_text(page)

if "_jlens_httpd" not in globals():
    _handler = partial(SimpleHTTPRequestHandler, directory=os.path.abspath("slices"))
    _jlens_httpd = HTTPServer(("127.0.0.1", 0), _handler)
    threading.Thread(target=_jlens_httpd.serve_forever, daemon=True).start()
print(f"-> http://localhost:{_jlens_httpd.server_address[1]}/{url}/")



## 4. Render a slice page (inline)

`compute_slice` + `build_page` produce an interactive position × layer view of the lens's token ranks (the `?` in the corner explains the controls). `mode="embed"` inlines everything so the page is self-contained.

In [ ]:
from jlens.vis import build_page, compute_slice, notebook_iframe

# import importlib
# importlib.reload(jlens)


reasoning_effort = 'medium'
example = example_es
lens = lens_original
max_tokens=1024

print(f"{reasoning_effort=}")
prompt = resolve_prompt_extended(example, tokenizer, enable_thinking=False, reasoning_effort=reasoning_effort)

full_text = get_model_response(prompt, max_tokens=max_tokens)
print("PROMPT: ", prompt)
print("\nRESPONSE: ", full_text)

slice_data = compute_slice(
    model,
    lens,
    prompt,
    # layer_stride=2,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
    last_n_tokens = 512,
    max_seq_len = 2000,
    response_prompt=full_text,
)


page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    # alt_token=gloss,
)
notebook_iframe(page, height = 800)

In [ ]:
# target_word = 'spiders'
# pos = 14
# layers = [14]


# reciprocal_ranks, ranks, final_rank, input_ids = get_concept_rr_across_layers(model, lens, prompt, target_word=target_word, position=pos, layers=layers)

prompt_spider = resolve_prompt_extended(example_en, tokenizer, enable_thinking=False, reasoning_effort="medium")
rr, ranks, final_rank, _ = get_concept_rr_across_layers(
    model, lens_original, prompt_spider, "spider", layers=[25], position=14
)
print("Layer 25 rank:", ranks[25])  # -> 1

rr

In [ ]:
example=example_es
prompt = resolve_prompt_extended(example, tokenizer, enable_thinking=False, reasoning_effort=reasoning_effort)

full_text = get_model_response(prompt, max_tokens=4000)
print("PROMPT: ", prompt)
print("\nRESPONSE: ", full_text)

In [ ]:
target_word = "spiders"
example = example_en
prompt = resolve_prompt_extended(example, tokenizer, enable_thinking=False, reasoning_effort=reasoning_effort)
reciprocal_ranks, ranks, final_rank, input_ids = get_concept_rr_across_layers(model, lens, prompt, target_word=target_word, position=14, layers=[25])
reciprocal_ranks

In [ ]:
# 1. Find all token positions overlapping the target words (returns list[int])
positions_en = find_word_position_in_example(
    example_en, "webs", tokenizer, enable_thinking=False, reasoning_effort=reasoning_effort
)  # e.g. [14]
positions_es = find_word_position_in_example(
    example_es, "telarañas", tokenizer, enable_thinking=False, reasoning_effort=reasoning_effort
)  # e.g. [16, 17, 18]

# Pick the last subword token position of each word (or pass the list directly)
pos_en = positions_en[-1]
pos_es = positions_es[-1]

# 2. Compare residual cosine distance, Recall@10, and target concept RR across layers
layers_of_interest = list(range(15, 31))
comparison = compare_examples_across_layers(
    model,
    lens_original,
    example_en=example_en,
    example_es=example_es,
    pos_en=pos_en,
    pos_es=pos_es,
    layers=layers_of_interest,
    target_word="spiders",
    k=10,
    mask_display=True,
    enable_thinking=False,
    reasoning_effort=reasoning_effort,
)

for l in layers_of_interest:
    print(
        f"L{l:02d} | raw_cos_dist={comparison.raw_cosine_distance[l]:.4f} "
        f"| J_cos_dist={comparison.transported_cosine_distance[l]:.4f} "
        f"| Recall@10={comparison.recall_at_k[l]:.2f} "
        f"| RR_en={comparison.rr_en[l]:.3f} (rank={comparison.ranks_en[l]}) "
        f"| RR_es={comparison.rr_es[l]:.3f} (rank={comparison.ranks_es[l]})"
    )

In [ ]:
reciprocal_ranks

In [ ]:
prompt = resolve_prompt_extended(example, tokenizer, enable_thinking=True, reasoning_effort=reasoning_effort)

full_text = get_model_response(prompt, max_tokens=3000)
print(full_text)

In [ ]:
from jlens.vis import build_page, compute_slice, notebook_iframe


prompt = resolve_prompt_extended(example, tokenizer, enable_thinking=False, reasoning_effort='low')

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
    last_n_tokens = 512,
)

full_text = get_model_response(prompt, max_tokens=800)
print("PROMPT: ", prompt)
print("\nRESPONSE: ", full_text)


page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    # alt_token=gloss,
)
notebook_iframe(page, height = 800)

In [ ]:
from jlens.vis import build_page, compute_slice, notebook_iframe


prompt = resolve_prompt_extended(example, tokenizer, enable_thinking=False, reasoning_effort='low')

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
    last_n_tokens = 512,
)

full_text = get_model_response(prompt, max_tokens=800)
print("PROMPT: ", prompt)
print("\nRESPONSE: ", full_text)


page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    # alt_token=gloss,
)
notebook_iframe(page, height = 800)

### More to explore

A few more prompts are bundled in `jlens.examples.EXAMPLES` — change the `slug` above and see what surfaces, or try a prompt of your own.

In [ ]:
for e in EXAMPLES:
    print(f"{e.slug:>24}  {e.section}")

## 6. Fitting

`fit(model, prompts)` computes `J_l` over the supplied prompts. 100 prompts is enough for a usable lens; the released lenses use 1000. `dim_batch` is the memory knob — each prompt does `ceil(d_model / dim_batch)` backward passes on a retained graph.

In [ ]:
from jlens.examples import load_wikitext_prompts
from datasets import load_dataset
import random
import numpy as np
import os
from transformers import set_seed
import torch



def set_seed_n(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    
    # These ensure deterministic behavior for certain algorithms
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # For Hugging Face transformers/datasets
    os.environ["PYTHONHASHSEED"] = str(seed)
    set_seed(seed)

set_seed_n(10)

dataset_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train", streaming=True)
dataset_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

dataset_en.shuffle()
dataset_es.shuffle()

prompts_en = load_wikitext_prompts(n_prompts=50, dataset=dataset_en)
prompts_es = load_wikitext_prompts(n_prompts=50, dataset=dataset_es)

prompts = prompts_en + prompts_es
random.shuffle(prompts)
fit_dim_batch = 8 if torch.cuda.is_available() else 1
lens = jlens.fit(
    model, prompts, dim_batch=fit_dim_batch, max_seq_len=128, checkpoint_path="ckpt.pt",
    checkpoint_every=1 
)
lens.save("jacobian_lens_50_en_50_es_from_scratch.pt")

In [ ]:
# Fit custom lens only in Spanish

from jlens.examples import load_wikitext_prompts
from datasets import load_dataset
import random
import numpy as np
import os
from transformers import set_seed
import torch

SEED = 10
set_seed(SEED)

# Number of prompts
N_PROMPTS = 64
model_name_spl = MODEL_NAME.split('/')[-1]  # Ex: Qwen3.5-9B
folder = f"jlens/computed/{model_name_spl}"
os.makedirs(folder, exist_ok=True)

relative_path = f"{folder}/{model_name_spl}_jpp_lens.pt"

dataset_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train", streaming=True)

dataset_es.shuffle()
prompts_es = load_wikitext_prompts(n_prompts=N_PROMPTS, dataset=dataset_es)

fit_dim_batch = 8 if torch.cuda.is_available() else 1
lens = jlens.fit(
    model, prompts_es, dim_batch=fit_dim_batch, max_seq_len=128, checkpoint_path="ckpt.pt",
    checkpoint_every=1 
)
lens.save(relative_path)
print(f"Lens saved to {relative_path}")

# Generate a README file with the configuration used
with open(f"{folder}/README.md", "w") as f:
    f.write(f"# Configuration for {model_name_spl} JPP Lens\n")
    f.write(f"Model name: {MODEL_NAME}\n")
    f.write(f"Dataset: {dataset_es}\n")
    f.write(f"Number of prompts: {N_PROMPTS}\n")
    f.write(f"Seed: {SEED}\n")
    f.write(f"Lens saved to: {relative_path}\n")

## 7. Ablation

In [ ]:
import importlib
importlib.reload(jlens.ablation)

In [ ]:
from jlens.ablation import ablate_jacobian_direction

device = next(hf_model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Identify the token direction to ablate (e.g., "Italy" / " Euro")
target_token_id = tokenizer.encode(" Italy", add_special_tokens=False)[0]
mid_to_late_layers = list(range(12, 28))

# 2. Baseline generation / forward pass
with torch.no_grad():
    base_logits = hf_model(**inputs).logits[:, -1, :]
    print("Baseline top-3:", tokenizer.batch_decode(base_logits.topk(3).indices[0]))

# 3. Forward pass with the J-lens direction removed across intermediate layers
with torch.no_grad():
    with ablate_jacobian_direction(hf_model, lens, target_token_id, layers=mid_to_late_layers):
        ablated_logits = hf_model(**inputs).logits[:, -1, :]
        print("Ablated top-3:", tokenizer.batch_decode(ablated_logits.topk(3).indices[0]))

In [ ]:
from datasets import load_dataset

# dataset = load_dataset(
#     "Salesforce/wikitext", "wikitext-103-raw-v1", split="train", streaming=True
# )

dataset = load_dataset("wikimedia/wikipedia", "20231101.es", split="train", streaming=True)

# prompts: list[str] = []
# for record in dataset:
#     text = record["text"]
#     if len(text.strip()) >= min_chars:
#         prompts.append(text)
#         if len(prompts) == n_prompts:
#             break
# return prompts


In [ ]:
dataset.shuffle()

In [ ]:
prompts[0]

## Swapping

Replacing one word in the J-space with another


In [ ]:
example=Example(
    slug="jorge_conversational",
    section="Jorge conversational",
    description="Jorge in conversational mode",
    user="What part of the body are you thinking of?",
    system="Respond in one word",
)

# Words to replace
src_word = 'Head'
dst_word = 'Foot'   

prompt = resolve_prompt_extended(example, tokenizer, enable_thinking=False, reasoning_effort='low')
prompt

# Other


In [ ]:
layers_of_interest = [21]
pos_en = 71
pos_es = 75


get_distance(
    model,
    lens_original,
    example_en=example_en,
    example_es=example_en,
    pos_en=pos_en,
    pos_es=pos_en,
    layers=layers_of_interest,
    mask_display=True,
    enable_thinking=False,
    reasoning_effort=reasoning_effort,
)

In [ ]:


prompt_es = resolve_prompt_extended(
        example_en,
        tokenizer,
        enable_thinking=False,
        reasoning_effort=reasoning_effort,
    )
get_token_position_after_think_block(tokenizer, prompt_es)

In [ ]:
reasoning_effort = 'medium'

# 1. Find all token positions overlapping the target words (returns list[int])
# positions_en = find_word_position_in_example(
#     example_en, "webs", tokenizer, enable_thinking=False, reasoning_effort=reasoning_effort
# )  # -> [14]
# positions_es = find_word_position_in_example(
#     example_es, "telarañas", tokenizer, enable_thinking=False, reasoning_effort=reasoning_effort
# )  # -> [16, 17, 18]

# # Pick the last subword token position of each word (or pass the list directly)
# pos_en = positions_en[-1]
# pos_es = positions_es[-1]

pos_en = 71 #-1
pos_es = 75 #-1


# 2. Compare residual cosine distance, Recall@10, and target concept RR across layers
layers_of_interest = list(range(21))
comparison = compare_examples_across_layers(
    model,
    lens_original,
    example_en=example_en,
    example_es=example_es,
    pos_en=pos_en,
    pos_es=pos_es,
    layers=layers_of_interest,
    target_word="spiders",
    k=10,
    mask_display=True,
    enable_thinking=False,
    reasoning_effort=reasoning_effort,
)

for l in layers_of_interest:
    print(
        f"L{l:02d} | raw_cos_dist={comparison.raw_cosine_distance[l]:.4f} "
        f"| J_cos_dist={comparison.transported_cosine_distance[l]:.4f} "
        f"| Recall@10={comparison.recall_at_k[l]:.2f} "
        f"| RR_en={comparison.rr_en[l]:.3f} (rank={comparison.ranks_en[l]}) "
        f"| RR_es={comparison.rr_es[l]:.3f} (rank={comparison.ranks_es[l]})"
    )